<a href="https://colab.research.google.com/github/marcilol/Healthcare_Daten/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# Import necessary libraries in Google Colab
import pandas as pd
import numpy as np
from random import randint, choice
from datetime import datetime, timedelta

# Data generation functions (copy from your `generate_data.py` script)
def generate_patients(num_patients=50000):
    patients = []
    for i in range(num_patients):
        patients.append({
            'patient_id': i,
            'age': randint(18, 90),
            'sex': choice(['M', 'F']),
            'race': choice(['White', 'Black', 'Asian', 'Hispanic', 'Other']),
            'zip_code': f'{randint(10000, 99999)}',
        })
    return pd.DataFrame(patients)

def generate_encounters(num_encounters=200000, num_patients=50000):
    encounters = []
    for i in range(num_encounters):
        encounters.append({
            'encounter_id': i,
            'patient_id': randint(0, num_patients - 1),
            'hospital_id': randint(0, 10),
            'admission_date': datetime.now() - timedelta(days=randint(1, 365)),
            'discharge_date': datetime.now() - timedelta(days=randint(1, 5)),
            'diagnosis': choice(['Diabetes', 'Hypertension', 'Pneumonia', 'Cancer', 'Heart Disease']),
            'length_of_stay': randint(1, 10),
            'visit_type': choice(['Emergency', 'Elective', 'Urgent', 'Routine']),
            'readmission_within_30_days': choice([0, 1])
        })
    return pd.DataFrame(encounters)


def generate_lab_results(NUM_LAB_RESULTS = 200000):
    """Generates synthetic lab result data."""
    lab_results = []
    for i in range(NUM_LAB_RESULTS):
        lab_results.append({
            'lab_id': i,
            'patient_id': randint(0, NUM_PATIENTS - 1),
            'test_name': choice(['Blood Pressure', 'Cholesterol', 'Glucose', 'Kidney Function']),
            'test_result': np.random.uniform(5, 200),
            'test_date': datetime.now() - timedelta(days=randint(1, 365))
        })
    return pd.DataFrame(lab_results)

def generate_medications(NUM_MEDICATIONS = 200000):
    """Generates synthetic medication data."""
    medications = []
    for i in range(NUM_MEDICATIONS):
        medications.append({
            'medication_id': i,
            'patient_id': randint(0, NUM_PATIENTS - 1),
            'medication_name': choice(['Metformin', 'Amlodipine', 'Lisinopril', 'Aspirin', 'Atorvastatin']),
            'dose': randint(1, 5),
            'frequency': choice(['Daily', 'Twice a day', 'As needed']),
        })
    return pd.DataFrame(medications)

def generate_and_save_data():
    """Generate and save synthetic data to CSV files."""
    patients = generate_patients()
    encounters = generate_encounters()
    lab_results = generate_lab_results()
    medications = generate_medications()

    patients.to_csv('data/raw/synthetic_data_patients.csv', index=False)
    encounters.to_csv('data/raw/synthetic_data_encounters.csv', index=False)
    lab_results.to_csv('data/raw/synthetic_data_lab_results.csv', index=False)
    medications.to_csv('data/raw/synthetic_data_medications.csv', index=False)

    print("Data generation complete.")

# Generate the data
patients = generate_patients()
encounters = generate_encounters()
lab_results = generate_lab_results()
medications = generate_medications()

# Save them as CSV files in Google Colab
patients.to_csv('/content/synthetic_data_patients.csv', index=False)
encounters.to_csv('/content/synthetic_data_encounters.csv', index=False)
lab_results.to_csv('/content/synthetic_data_lab_results.csv', index=False)
medications.to_csv('/content/synthetic_data_medications.csv', index=False)


In [28]:
# src/etl/etl_pipeline.py
import pandas as pd
import sqlite3
from config import DATABASE_PATH

def create_tables(conn):
    """Creates tables for storing patient, encounter, lab results, and medication data."""
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS patients (
        patient_id INTEGER PRIMARY KEY,
        age INTEGER,
        sex TEXT,
        race TEXT,
        zip_code TEXT
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS encounters (
        encounter_id INTEGER PRIMARY KEY,
        patient_id INTEGER,
        hospital_id INTEGER,
        admission_date TEXT,
        discharge_date TEXT,
        diagnosis TEXT,
        length_of_stay INTEGER,
        visit_type TEXT,
        readmission_within_30_days INTEGER,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS lab_results (
        lab_id INTEGER PRIMARY KEY,
        patient_id INTEGER,
        test_name TEXT,
        test_result REAL,
        test_date TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS medications (
        medication_id INTEGER PRIMARY KEY,
        patient_id INTEGER,
        medication_name TEXT,
        dose INTEGER,
        frequency TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
    )
    ''')

    conn.commit()

def load_data_into_db():
    """Load generated data into SQLite database."""
    patients = pd.read_csv('data/raw/synthetic_data_patients.csv')
    encounters = pd.read_csv('data/raw/synthetic_data_encounters.csv')
    lab_results = pd.read_csv('data/raw/synthetic_data_lab_results.csv')
    medications = pd.read_csv('data/raw/synthetic_data_medications.csv')

    conn = sqlite3.connect(DATABASE_PATH)
    create_tables(conn)

    patients.to_sql('patients', conn, if_exists='replace', index=False)
    encounters.to_sql('encounters', conn, if_exists='replace', index=False)
    lab_results.to_sql('lab_results', conn, if_exists='replace', index=False)
    medications.to_sql('medications', conn, if_exists='replace', index=False)

    conn.close()
    print("Data loaded into database.")

ModuleNotFoundError: No module named 'config'

In [32]:
import sqlite3

# Connect to SQLite database in Colab
conn = sqlite3.connect('/content/healthcare_data.db')
cursor = conn.cursor()

# Create tables in the database (use your ETL pipeline logic)
cursor.execute('''
CREATE TABLE IF NOT EXISTS patients (
    patient_id INTEGER PRIMARY KEY,
    age INTEGER,
    sex TEXT,
    race TEXT,
    zip_code TEXT
)
''')
cursor.execute('''
CREATE TABLE IF NOT EXISTS encounters (
    encounter_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    hospital_id INTEGER,
    admission_date TEXT,
    discharge_date TEXT,
    diagnosis TEXT,
    length_of_stay INTEGER,
    visit_type TEXT,
    readmission_within_30_days INTEGER,
    FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
)
''')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS lab_results (
        lab_id INTEGER PRIMARY KEY,
        patient_id INTEGER,
        test_name TEXT,
        test_result REAL,
        test_date TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
    )
    ''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS medications (
        medication_id INTEGER PRIMARY KEY,
        patient_id INTEGER,
        medication_name TEXT,
        dose INTEGER,
        frequency TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
    )
    ''')

# Insert synthetic data into the SQLite database
patients_df = pd.read_csv('/content/synthetic_data_patients.csv')
encounters_df = pd.read_csv('/content/synthetic_data_encounters.csv')
lab_results = pd.read_csv('/content/synthetic_data_lab_results.csv')
medications = pd.read_csv('/content/synthetic_data_medications.csv')

patients_df.to_sql('patients', conn, if_exists='replace', index=False)
encounters_df.to_sql('encounters', conn, if_exists='replace', index=False)
lab_results.to_sql('lab_results', conn, if_exists='replace', index=False)
medications.to_sql('medications', conn, if_exists='replace', index=False)

# Commit and close the database connection
conn.commit()
conn.close()
print("Data loaded into database.")

Data loaded into database.


In [33]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import sqlite3
import joblib

# Load data from SQLite
conn = sqlite3.connect('/content/healthcare_data.db')
query = '''
SELECT e.patient_id, p.age, e.diagnosis, e.length_of_stay, e.readmission_within_30_days,
       COUNT(e.encounter_id) AS prior_visits,
       MAX(l.test_result) AS max_lab_result
FROM encounters e
JOIN patients p ON e.patient_id = p.patient_id
LEFT JOIN lab_results l ON e.patient_id = l.patient_id
GROUP BY e.patient_id
'''
data = pd.read_sql(query, conn)

# Data Preprocessing
data['diagnosis'] = data['diagnosis'].map({
    'Diabetes': 1, 'Hypertension': 2, 'Pneumonia': 3, 'Cancer': 4, 'Heart Disease': 5
})
X = data[['age', 'diagnosis', 'length_of_stay', 'prior_visits', 'max_lab_result']]
y = data['readmission_within_30_days']

# Train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Save the trained model
joblib.dump(clf, '/content/readmission_model.pkl')

              precision    recall  f1-score   support

           0       0.50      0.51      0.51      4888
           1       0.51      0.50      0.50      4934

    accuracy                           0.50      9822
   macro avg       0.50      0.50      0.50      9822
weighted avg       0.50      0.50      0.50      9822



['/content/readmission_model.pkl']

In [19]:
# src/data_generation/generate_data.py
import pandas as pd
import numpy as np
from random import randint, choice
from datetime import datetime, timedelta
NUM_PATIENTS = 50000
NUM_ENCOUNTERS = 200000
NUM_LAB_RESULTS = 200000
NUM_MEDICATIONS = 200000
def generate_patients():
    """Generates synthetic patient data."""
    patients = []
    for i in range(NUM_PATIENTS):
        patients.append({
            'patient_id': i,
            'age': randint(18, 90),
            'sex': choice(['M', 'F']),
            'race': choice(['White', 'Black', 'Asian', 'Hispanic', 'Other']),
            'zip_code': f'{randint(10000, 99999)}',
        })
    return pd.DataFrame(patients)

def generate_encounters():
    """Generates synthetic hospital encounter data."""
    encounters = []
    for i in range(NUM_ENCOUNTERS):
        encounters.append({
            'encounter_id': i,
            'patient_id': randint(0, NUM_PATIENTS - 1),
            'hospital_id': randint(0, 10),
            'admission_date': datetime.now() - timedelta(days=randint(1, 365)),
            'discharge_date': datetime.now() - timedelta(days=randint(1, 5)),
            'diagnosis': choice(['Diabetes', 'Hypertension', 'Pneumonia', 'Cancer', 'Heart Disease']),
            'length_of_stay': randint(1, 10),
            'visit_type': choice(['Emergency', 'Elective', 'Urgent', 'Routine']),
            'readmission_within_30_days': choice([0, 1])
        })
    return pd.DataFrame(encounters)

def generate_lab_results():
    """Generates synthetic lab result data."""
    lab_results = []
    for i in range(NUM_LAB_RESULTS):
        lab_results.append({
            'lab_id': i,
            'patient_id': randint(0, NUM_PATIENTS - 1),
            'test_name': choice(['Blood Pressure', 'Cholesterol', 'Glucose', 'Kidney Function']),
            'test_result': np.random.uniform(5, 200),
            'test_date': datetime.now() - timedelta(days=randint(1, 365))
        })
    return pd.DataFrame(lab_results)

def generate_medications():
    """Generates synthetic medication data."""
    medications = []
    for i in range(NUM_MEDICATIONS):
        medications.append({
            'medication_id': i,
            'patient_id': randint(0, NUM_PATIENTS - 1),
            'medication_name': choice(['Metformin', 'Amlodipine', 'Lisinopril', 'Aspirin', 'Atorvastatin']),
            'dose': randint(1, 5),
            'frequency': choice(['Daily', 'Twice a day', 'As needed']),
        })
    return pd.DataFrame(medications)

def generate_and_save_data():
    """Generate and save synthetic data to CSV files."""
    patients = generate_patients()
    encounters = generate_encounters()
    lab_results = generate_lab_results()
    medications = generate_medications()

    patients.to_csv('data/raw/synthetic_data_patients.csv', index=False)
    encounters.to_csv('data/raw/synthetic_data_encounters.csv', index=False)
    lab_results.to_csv('data/raw/synthetic_data_lab_results.csv', index=False)
    medications.to_csv('data/raw/synthetic_data_medications.csv', index=False)

    print("Data generation complete.")

In [13]:
# config.py
import os

# Database Configuration
DATABASE_PATH = os.path.join(os.getcwd(), 'healthcare_data.db')

# Model Configuration
MODEL_PATH = os.path.join(os.getcwd(), 'models', 'readmission_model.pkl')

# Data Generation Parameters
NUM_PATIENTS = 50000
NUM_ENCOUNTERS = 200000
NUM_LAB_RESULTS = 200000
NUM_MEDICATIONS = 200000